# Notebook 1: Simulación de los Escenarios
## Genera el dataset CSV de bits (Alice / Eve / Bob) para cada escenario industrial

**Entradas:** 12 archivos JSON con los parámetros de cada escenario.  
**Salida:** Un CSV por escenario con columnas `distancia, experimento, n_bit, bit_alice, bit_eve, bit_bob`.

### Física implementada
- Fuente real Poisson Alice: $n \sim \text{Poisson}(\mu)$ fotones por pulso. Pulsos vacío ($n=0$) descartados.
- Fuente real Poisson Eve: $m \sim \text{Poisson}(\mu_{\text{Eve}})$ fotones por pulso reenviado. Si $m=0$, Eve no reenvía nada (absorción total del pulso).
- Codificación en polarización: estado cuántico $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$.
- Errores estocásticos: bit flip (puerta $X$) y phase flip (puerta $Z$) con $p(L) = p_0 + p_{\text{dist}}(1-e^{-L/L_s})$.
- Si hay Eve: ruido **antes y después** del intercept-resend.
- Detección: transmitancia $\eta_{ch}=10^{-\alpha L/10}$, eficiencia $\eta_{det}$, dark counts, descarte de doble click.
- Sifting: bases Alice = Bob **y** exactamente 1 click en Bob.


## 0. Imports

In [1]:
import numpy as np
import pandas as pd
import json, os, glob
from pathlib import Path

## 1. Parámetros globales de la simulación

In [2]:
# ── Parámetros comunes a todos los escenarios ──────────────────────────────
N_PULSES     = 100_000              # Pulsos enviados por Alice por (semilla × distancia)
DISTANCES_KM = list(range(0, 101, 5))  # Distancias (km)
SEEDS        = [42, 123, 256, 789, 1024, 2048, 4096, 8192] 

# ── Rutas ──────────────────────────────────────────────────────────────────
JSON_DIR  = "scenarios"          # Directorio con los 12 JSON de escenarios
OUTPUT_DIR = "output_csv"  # Directorio de salida de CSV
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Funciones cuánticas y de canal

In [3]:
s2 = 1.0 / np.sqrt(2.0)

# ── Codificación de qubit ──────────────────────────────────────────────────
# Devuelve (N,2) array de amplitudes [α, β]
def encode_qubits(bits: np.ndarray, bases_z: np.ndarray) -> np.ndarray:
    """Codifica N bits en estados cuánticos según base Z/X."""
    states = np.empty((len(bits), 2), dtype=float)
    states[ bases_z  & (bits == 0)] = [1.0,  0.0]
    states[ bases_z  & (bits == 1)] = [0.0,  1.0]
    states[~bases_z  & (bits == 0)] = [s2,   s2 ]
    states[~bases_z  & (bits == 1)] = [s2,  -s2 ]
    return states


def measure_prob0(states: np.ndarray, bases_z: np.ndarray) -> np.ndarray:
    """Probabilidad cuántica de obtener 0 al medir en base Z o X."""
    alpha, beta = states[:, 0], states[:, 1]
    p_z = alpha ** 2
    p_x = ((alpha + beta) * s2) ** 2
    return np.clip(np.where(bases_z, p_z, p_x), 0.0, 1.0)


def quantum_measure(states: np.ndarray, bases_z: np.ndarray,
                    rng: np.random.Generator) -> np.ndarray:
    """Colapso cuántico: retorna array de bits medidos."""
    return (rng.random(len(states)) > measure_prob0(states, bases_z)).astype(int)


# ── Errores de canal ───────────────────────────────────────────────────────
def error_prob(p0, pdist, lscale, dist):
    return p0 + pdist * (1.0 - np.exp(-dist / lscale)) if lscale > 0 else p0 + pdist


def apply_noise(states: np.ndarray, p_bf: float, p_pf: float,
                rng: np.random.Generator) -> np.ndarray:
    """Aplica bit flip (puerta X) y phase flip (puerta Z) estocásticamente."""
    st = states.copy()
    if p_bf > 0:
        mask = rng.random(len(st)) < p_bf
        st[mask, 0], st[mask, 1] = st[mask, 1].copy(), st[mask, 0].copy()
    if p_pf > 0:
        mask = rng.random(len(st)) < p_pf
        st[mask, 1] = -st[mask, 1]
    return st


# ── Canal óptico ───────────────────────────────────────────────────────────
def channel_eta(fiber_loss_db, dist):
    return 10 ** (-fiber_loss_db * dist / 10)


## 3. Simulación BB84 para un (escenario, semilla, distancia)

In [4]:
def simulate_bb84(sc: dict, dist: float, seed: int) -> pd.DataFrame:
    """
    Simula el protocolo BB84 completo.

    Retorna DataFrame con columnas:
        distancia, experimento (=seed), n_bit, bit_alice, bit_eve, bit_bob
    donde bit_eve es NaN si el pulso no fue interceptado.
    """
    rng = np.random.default_rng(seed)
    N   = N_PULSES

    # ── 1. Alice ──────────────────────────────────────────────────────────
    alice_bits  = rng.integers(0, 2, N)
    alice_bases = rng.random(N) < sc["prob_base_z_alice"]
    states      = encode_qubits(alice_bits, alice_bases)

    # ── 2. Fuente Poisson; descartar pulsos vacío ─────────────────────────
    n_photons   = rng.poisson(sc["mu"], N)
    alice_sent  = n_photons >= 1            # solo pulsos con ≥1 fotón

    # ── 3. Ruido ANTES de Eve ─────────────────────────────────────────────
    p_bf = error_prob(sc["bit_flip_p0"],   sc["bit_flip_pdist"],   sc["bit_flip_lscale"],   dist)
    p_pf = error_prob(sc["phase_flip_p0"], sc["phase_flip_pdist"], sc["phase_flip_lscale"], dist)
    states = apply_noise(states, p_bf, p_pf, rng)

    # ── 4. Ataque intercept-resend de Eve ─────────────────────────────────
    eve_frac   = sc["eve_fraction"]
    eve_mask   = np.zeros(N, dtype=bool)
    eve_bits   = np.full(N, np.nan)
    n_intercept = int(round(eve_frac * N))
    if n_intercept > 0:
        idx = rng.choice(N, size=n_intercept, replace=False)
        eve_mask[idx] = True

    states_after_eve = states.copy()
    n_photons_eff    = n_photons.copy()

    if eve_mask.any():
        eve_bases_arr = rng.random(N) < sc["prob_base_z_eve"]
        # Eve mide (colapso cuántico)
        eve_measured = quantum_measure(states, eve_bases_arr, rng)

        # ── Fuente real de Eve: Poisson(mu_eve) fotones reenviados ────────
        # Eve mide el pulso entrante y re-prepara un nuevo estado cuántico,
        # pero su fuente no es ideal: emite m ~ Poisson(mu_eve) fotones.
        # Si m = 0, Eve absorbe el pulso (no llega nada a Bob).
        mu_eve       = sc["mu_eve"]
        n_eve_resend = rng.poisson(mu_eve, N)           # fotones reenviados por Eve
        eve_resends  = eve_mask & (n_eve_resend >= 1)   # pulsos que Eve reenvía efectivamente

        # Eve registra su bit solo en pulsos interceptados con fotón de Alice
        eve_bits = np.where(eve_mask & alice_sent, eve_measured.astype(float), np.nan)

        # Re-preparar estado cuántico para los pulsos que Eve sí reenvía
        for i in np.where(eve_resends)[0]:
            states_after_eve[i] = encode_qubits(
                np.array([eve_measured[i]]),
                np.array([bool(eve_bases_arr[i])])
            )[0]

        # Número efectivo de fotones que viajan hacia Bob:
        #  - Pulsos interceptados y reenviados: n_eve_resend (Poisson)
        #  - Pulsos interceptados pero absorbidos (m=0): 0 fotones → Bob no detecta
        #  - Pulsos no interceptados: n_photons original de Alice
        n_photons_eff = n_photons.copy()
        n_photons_eff[eve_mask] = n_eve_resend[eve_mask]

    # ── 5. Ruido DESPUÉS de Eve ───────────────────────────────────────────
    states_after_eve = apply_noise(states_after_eve, p_bf, p_pf, rng)

    # ── 6. Transmitancia y detección en Bob ───────────────────────────────
    eta_ch    = channel_eta(sc["fiber_loss_db_per_km"], dist)
    eta_total = eta_ch * sc["eta_detector"]
    nf        = n_photons_eff.astype(float)

    #p_real   = 1.0 - np.exp(-eta_total * nf)
    p_real = 1.0 - (1.0 - eta_total) ** nf
    real_clk = rng.random(N) < p_real

    # Doble click: P(≥2 clicks reales)
    nf_safe  = np.maximum(nf - 1, 0)
    p_dbl    = np.where(nf >= 2,
                        1.0 - (1-eta_total)**nf - nf*eta_total*(1-eta_total)**nf_safe,
                        0.0)
    dbl_clk  = rng.random(N) < p_dbl
    drk_clk  = rng.random(N) < sc["dark_count_prob"]

    total_clk = real_clk | drk_clk
    # Sifting: exactamente 1 click (no doble click) y pulso válido de Alice
    valid_clk = total_clk & ~dbl_clk

    # ── 7. Bob mide ───────────────────────────────────────────────────────
    bob_bases = rng.random(N) < sc["prob_base_z_bob"]
    bob_measured = quantum_measure(states_after_eve, bob_bases, rng)
    # Dark count sin fotón real → bit aleatorio
    dark_only = drk_clk & ~real_clk
    bob_bits  = np.where(dark_only, rng.integers(0, 2, N), bob_measured)

    # ── 8. Sifting: base coincidente + 1 click + Alice envió fotón ────────
    same_basis  = (alice_bases == bob_bases)
    sifted_mask = valid_clk & same_basis & alice_sent

    idx_sifted = np.where(sifted_mask)[0]
    n_sifted   = len(idx_sifted)

    if n_sifted == 0:
        return pd.DataFrame(columns=["distancia","experimento","n_bit",
                                     "bit_alice","bit_eve","bit_bob"])

    df = pd.DataFrame({
        "distancia"  : dist,
        "experimento": seed,
        "n_bit"      : np.arange(1, n_sifted + 1),
        "bit_alice"  : alice_bits[idx_sifted],
        "bit_eve"    : eve_bits[idx_sifted],
        "bit_bob"    : bob_bits[idx_sifted],
    })
    return df


## 4. Bucle principal: escenarios × semillas × distancias

In [5]:
json_files = sorted(glob.glob(os.path.join(JSON_DIR, "scenario_*.json")))
if not json_files:
    raise FileNotFoundError(f"No se encontraron JSON en '{JSON_DIR}'. "
                            "Ajusta JSON_DIR o copia los archivos allí.")

print(f"Escenarios encontrados: {len(json_files)}")
print(f"Semillas: {SEEDS}")
print(f"Distancias (km): {DISTANCES_KM}")
print(f"Pulsos por ejecución: {N_PULSES:,}\n")

for jpath in json_files:
    with open(jpath) as f:
        sc = json.load(f)

    sc_id   = sc["id"]
    sc_name = sc["name"]
    print(f"─── Escenario {sc_id:02d}: {sc_name} ───")

    parts = []
    for dist in DISTANCES_KM:
        for seed in SEEDS:
            df_part = simulate_bb84(sc, dist, seed)
            parts.append(df_part)

    df_all = pd.concat(parts, ignore_index=True)

    out_name = f"scenario_{sc_id:02d}_{sc_name.lower().replace(' ','_')}.csv"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    df_all.to_csv(out_path, index=False)
    n_rows = len(df_all)
    print(f"  Guardado: {out_path}  ({n_rows:,} filas)\n")

print("✓ Todos los CSV generados.")


Escenarios encontrados: 12
Semillas: [42, 123, 256, 789, 1024, 2048, 4096, 8192]
Distancias (km): [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
Pulsos por ejecución: 100,000

─── Escenario 01: Ideal ───
  Guardado: output_csv\scenario_01_ideal.csv  (1,198,169 filas)

─── Escenario 02: Canal con ruido ───
  Guardado: output_csv\scenario_02_canal_con_ruido.csv  (1,196,558 filas)

─── Escenario 03: Canal con ruido alto ───
  Guardado: output_csv\scenario_03_canal_con_ruido_alto.csv  (1,196,558 filas)

─── Escenario 04: Ataque Eve parcial ───
  Guardado: output_csv\scenario_04_ataque_eve_parcial.csv  (1,110,129 filas)

─── Escenario 05: Ataque Eve total ───
  Guardado: output_csv\scenario_05_ataque_eve_total.csv  (754,425 filas)

─── Escenario 06: Fibra deteriorada ───
  Guardado: output_csv\scenario_06_fibra_deteriorada.csv  (529,725 filas)

─── Escenario 07: Detector ruidoso ───
  Guardado: output_csv\scenario_07_detector_ruidoso.csv  (1,235,858 fila

## 5. Vista previa del último CSV generado

In [6]:
print(f"Muestra de: {out_path}")
display(df_all.head(20))
print(f"\nColumnas: {list(df_all.columns)}")
print(f"Distancias: {sorted(df_all['distancia'].unique())}")
print(f"Experimentos (semillas): {sorted(df_all['experimento'].unique())}")
print(f"Bits con Eve activa: {df_all['bit_eve'].notna().sum():,}")


Muestra de: output_csv\scenario_12_escenario_adverso_3.csv


,distancia,experimento,n_bit,bit_alice,bit_eve,bit_bob
0,0,42,1,1,NaN,1
1,0,42,2,1,NaN,1
2,0,42,3,1,NaN,1
3,0,42,4,0,NaN,0
4,0,42,5,1,NaN,1
5,0,42,6,0,NaN,0
6,0,42,7,0,NaN,0
7,0,42,8,0,NaN,0
8,0,42,9,1,NaN,1
9,0,42,10,1,NaN,1



Columnas: ['distancia', 'experimento', 'n_bit', 'bit_alice', 'bit_eve', 'bit_bob']
Distancias: [np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(30), np.int64(35), np.int64(40), np.int64(45), np.int64(50), np.int64(55), np.int64(60), np.int64(65), np.int64(70), np.int64(75), np.int64(80), np.int64(85), np.int64(90), np.int64(95), np.int64(100)]
Experimentos (semillas): [np.int64(42), np.int64(123), np.int64(256), np.int64(789), np.int64(1024), np.int64(2048), np.int64(4096), np.int64(8192)]
Bits con Eve activa: 66,413
